# 02 — Load Raw (Parquet Snapshots to Volume)
Extracts from source via federation and writes as immutable Parquet snapshots
into the raw volume. Each run creates a new timestamped folder — never overwrites.

In [0]:
# Widget parameters
dbutils.widgets.text("catalog_name", "workspace", "Catalog Name")
dbutils.widgets.text("schema_name", "raw_zone", "Schema Name")
dbutils.widgets.text("base_path", "/Volumes/workspace/raw_zone/chinook", "Base Path")
dbutils.widgets.text("table_name", "all", "Table Name")
dbutils.widgets.text("run_id", "", "Run ID (auto-generated if blank)")

catalog = dbutils.widgets.get("catalog_name")
run_id_param = dbutils.widgets.get("run_id")

In [0]:
from datetime import datetime
import uuid

# Get run_id from upstream task or generate new
try:
    run_id = dbutils.jobs.taskValues.get(taskKey="task_01_extract_from_source", key="run_id")
except:
    run_id = run_id_param if run_id_param else str(uuid.uuid4())[:8]

execution_ts = datetime.now()
ts_path = execution_ts.strftime("%Y/%m/%d/%H%M%S")
base_volume_path = f"/Volumes/{catalog}/raw_zone/chinook"

print(f"Run ID: {run_id}")
print(f"Raw volume path: {base_volume_path}")
print(f"Timestamp path segment: {ts_path}")

Run ID: 5fe134f9
Raw volume path: /Volumes/workspace/raw_zone/chinook
Timestamp path segment: 2026/03/28/053151


## 1. Get Active Tables from Metadata

In [0]:
df_metadata = spark.sql(f"""
    SELECT table_name, source_catalog, source_schema, source_table, file_name, primary_key_col
    FROM {catalog}.metadata.pipeline_parent_metadata
    WHERE active_flag = 'Y'
    ORDER BY load_sequence
""")

tables = df_metadata.collect()
print(f"Tables to load to raw: {len(tables)}")

Tables to load to raw: 11


## 2. Extract from Source and Write Parquet Snapshots

Path pattern: `/Volumes/workspace/raw_zone/chinook/<table_name>/YYYY/MM/DD/HHMMSS/<table_name>.parquet`

In [0]:
load_results = []

for row in tables:
    start_time = datetime.now()
    table_name = row.table_name
    source_catalog = row.source_catalog
    source_schema = row.source_schema
    source_table = row.source_table

    try:
        # Read from source via Lakehouse Federation
        source_fqn = f"{source_catalog}.{source_schema}.{source_table}"
        df = spark.read.table(source_fqn)
        source_count = df.count()

        # Build immutable raw path
        raw_file_path = f"{base_volume_path}/{table_name}/{ts_path}/{table_name}.parquet"
        print(f"\nWriting: {table_name} → {raw_file_path}")

        # Write Parquet snapshot (new path each run — immutable)
        df.write.mode("overwrite").parquet(raw_file_path)

        # Verify target count
        df_verify = spark.read.parquet(raw_file_path)
        target_count = df_verify.count()

        end_time = datetime.now()
        duration = (end_time - start_time).total_seconds()

        # Validate counts match
        status = "success" if source_count == target_count else "count_mismatch"
        symbol = "✓" if status == "success" else "⚠"
        print(f"  {symbol} {table_name}: source={source_count}, raw={target_count} ({duration:.1f}s)")

        load_results.append({
            "table_name": table_name,
            "source_count": source_count,
            "target_count": target_count,
            "file_path": raw_file_path,
            "status": status,
            "duration": duration
        })

        # Log to child metrics
        spark.sql(f"""
            INSERT INTO {catalog}.metadata.pipeline_child_metrics
            VALUES (
                '{run_id}', '{table_name}', 'load_raw',
                current_timestamp(), '{status}',
                {source_count}, {target_count},
                NULL, NULL,
                '{raw_file_path}', NULL,
                '{start_time}', '{end_time}', {duration},
                current_timestamp()
            )
        """)

    except Exception as e:
        end_time = datetime.now()
        duration = (end_time - start_time).total_seconds()
        error_msg = str(e).replace("'", "''")[:500]
        print(f"  ✗ {table_name}: FAILED — {str(e)[:200]}")

        load_results.append({
            "table_name": table_name,
            "source_count": 0,
            "target_count": 0,
            "file_path": "",
            "status": "failed",
            "duration": duration
        })

        spark.sql(f"""
            INSERT INTO {catalog}.metadata.pipeline_child_metrics
            VALUES (
                '{run_id}', '{table_name}', 'load_raw',
                current_timestamp(), 'failed',
                0, 0,
                NULL, NULL, '', '{error_msg}',
                '{start_time}', '{end_time}', {duration},
                current_timestamp()
            )
        """)


Writing: album → /Volumes/workspace/raw_zone/chinook/album/2026/03/28/053151/album.parquet
  ✓ album: source=347, raw=347 (7.5s)

Writing: artist → /Volumes/workspace/raw_zone/chinook/artist/2026/03/28/053151/artist.parquet
  ✓ artist: source=275, raw=275 (4.6s)

Writing: customer → /Volumes/workspace/raw_zone/chinook/customer/2026/03/28/053151/customer.parquet
  ✓ customer: source=59, raw=59 (4.5s)

Writing: employee → /Volumes/workspace/raw_zone/chinook/employee/2026/03/28/053151/employee.parquet
  ✓ employee: source=8, raw=8 (4.8s)

Writing: genre → /Volumes/workspace/raw_zone/chinook/genre/2026/03/28/053151/genre.parquet
  ✓ genre: source=25, raw=25 (4.8s)

Writing: invoice → /Volumes/workspace/raw_zone/chinook/invoice/2026/03/28/053151/invoice.parquet
  ✓ invoice: source=412, raw=412 (5.0s)

Writing: invoiceline → /Volumes/workspace/raw_zone/chinook/invoiceline/2026/03/28/053151/invoiceline.parquet
  ✓ invoiceline: source=2240, raw=2240 (4.4s)

Writing: mediatype → /Volumes/works

## 3. Raw Load Summary

In [0]:
import pandas as pd

summary_df = pd.DataFrame(load_results)
print(f"\n{'='*60}")
print(f"RAW LOAD SUMMARY — Run ID: {run_id}")
print(f"{'='*60}")
print(f"Total tables:    {len(load_results)}")
print(f"Successful:      {sum(1 for r in load_results if r['status'] == 'success')}")
print(f"Count mismatch:  {sum(1 for r in load_results if r['status'] == 'count_mismatch')}")
print(f"Failed:          {sum(1 for r in load_results if r['status'] == 'failed')}")
print(f"{'='*60}")

display(spark.createDataFrame(summary_df))


RAW LOAD SUMMARY — Run ID: 5fe134f9
Total tables:    11
Successful:      11
Count mismatch:  0
Failed:          0


table_name,source_count,target_count,file_path,status,duration
album,347,347,/Volumes/workspace/raw_zone/chinook/album/2026/03/28/053151/album.parquet,success,7.455292
artist,275,275,/Volumes/workspace/raw_zone/chinook/artist/2026/03/28/053151/artist.parquet,success,4.575743
customer,59,59,/Volumes/workspace/raw_zone/chinook/customer/2026/03/28/053151/customer.parquet,success,4.513914
employee,8,8,/Volumes/workspace/raw_zone/chinook/employee/2026/03/28/053151/employee.parquet,success,4.755514
genre,25,25,/Volumes/workspace/raw_zone/chinook/genre/2026/03/28/053151/genre.parquet,success,4.803063
invoice,412,412,/Volumes/workspace/raw_zone/chinook/invoice/2026/03/28/053151/invoice.parquet,success,4.951452
invoiceline,2240,2240,/Volumes/workspace/raw_zone/chinook/invoiceline/2026/03/28/053151/invoiceline.parquet,success,4.41556
mediatype,5,5,/Volumes/workspace/raw_zone/chinook/mediatype/2026/03/28/053151/mediatype.parquet,success,4.542713
playlist,18,18,/Volumes/workspace/raw_zone/chinook/playlist/2026/03/28/053151/playlist.parquet,success,4.043675
playlisttrack,8715,8715,/Volumes/workspace/raw_zone/chinook/playlisttrack/2026/03/28/053151/playlisttrack.parquet,success,4.183409


## 4. Verify Volume Structure

In [0]:
# Show raw volume contents
for t in [r["table_name"] for r in load_results if r["status"] == "success"]:
    print(f"  /{t}/")
print(f"\nFull path example: {load_results[0]['file_path'] if load_results else 'N/A'}")

  /album/
  /artist/
  /customer/
  /employee/
  /genre/
  /invoice/
  /invoiceline/
  /mediatype/
  /playlist/
  /playlisttrack/
  /track/

Full path example: /Volumes/workspace/raw_zone/chinook/album/2026/03/28/053151/album.parquet


In [0]:
# Store values for downstream
dbutils.jobs.taskValues.set(key="run_id", value=run_id)
dbutils.jobs.taskValues.set(key="ts_path", value=ts_path)
print(f"Task values set — run_id: {run_id}, ts_path: {ts_path}")

Task values set — run_id: 5fe134f9, ts_path: 2026/03/28/053151
